In [1]:
import os

#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

!pwd
os.chdir("../RecSys_Course_AT_PoliMi")
!pwd

#!python run_compile_all_cython.py

/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/hybrids_testing
/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi


In [2]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from scipy.sparse import csr_matrix
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
#from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

from Recommenders.MatrixFactorization.IALSRecommender import FeatureCombinedImplicitALSRecommender

from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender

from Recommenders.hybrid.m2_model import IntegratedHierarchicalHybridRecommender
from Recommenders.hybrid.LinearHybridRecommender import GeneralizedLinearCoupleHybridRecommender


Tensorflow is not available


In [3]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

# Training knn costum similarity

In [4]:
SLIM_params = {
    'topK': 625,
    'l1_ratio': 0.09517221375634205,
    'alpha': 0.00228698730766055
}

ease_params = {
    'topK': 1431,
    'l2_norm': 426.57622242296605
}

"""rp3_params = {
    'alpha': 0.7733352330682174,
    'beta': 0.4139018623121251,
    'topK': 35
}"""

IALS_params = {
    'iterations': 135,
    'factors': 87,
    'alpha': 7.762338288061237,
    'regularization': 0.004799745261257595
}

In [5]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))


In [6]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [7]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [8]:
prefitted_folds = []

for i in range(5):
    print(f"Fitting fold {i+1}/5...")
    
    # Creazione URM Train (Combined) e Test per questo fold
    URM_train = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
    URM_test = URM_parts[i]
    
    # 1. Fit EASE
    recommender_ease = EASE_R_Recommender(URM_train)
    recommender_ease.fit(**ease_params)

    # 2. Fit SLIM
    recommender_SLIM = SLIMElasticNetRecommender(URM_train)
    recommender_SLIM.fit(**SLIM_params)
    
    # 3. Fit IALS
    als_recommender = FeatureCombinedImplicitALSRecommender(URM_train)
    als_recommender.fit(**IALS_params)
    
    # Prepariamo l'evaluator per questo fold specifico
    evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[20])
    
    # Salviamo tutto in un dizionario per questo fold
    fold_data = {
        "URM_train": URM_train,
        "ease": recommender_ease,
        "slim": recommender_SLIM,
        "ials": als_recommender,
        "evaluator": evaluator_test
    }
    
    prefitted_folds.append(fold_data)

print("Pre-training completato.")

Fitting fold 1/5...
EASE_R_Recommender: Fitting model... 


KeyboardInterrupt: 

In [ ]:

import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):

    start_time = time.time()
    scores = []
    for fold_data in prefitted_folds:
        
        # Recuperiamo i modelli e i dati dal dizionario
        URM_train = fold_data["URM_train"]
        recommender_ease = fold_data["ease"]
        recommender_SLIM = fold_data["slim"]
        als_recommender = fold_data["ials"]
        evaluator_test = fold_data["evaluator"]
        
        
        recommender = IntegratedHierarchicalHybridRecommender(
            URM_train, 
            recommender_ease, 
            recommender_SLIM, 
            als_recommender
        )
        
        alpha=optuna_trial.suggest_float("alpha", 0.60, 0.80)
        beta=optuna_trial.suggest_float("beta", 0.80, 1)
        
        recommender.fit(alpha, beta)
        
        
        result, _ = evaluator_test.evaluateRecommender(recommender)
        #print("prova = ", result["MAP"].values[0])
        #print(i)
        #print(result["RECALL"])
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)


In [ ]:
import optuna

optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 100)

[I 2025-12-24 19:29:22,269] A new study created in memory with name: no-name-b2c91888-c964-4aff-b4d9-dcf517c3dc8d


IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.6004642792452902...
IntegratedHierarchicalHybridRecommender: Fitting complete. Ready to score with beta=0.8911991669986515.
EvaluatorHoldout: Processed 27062 (100.0%) in 12.23 sec. Users per second: 2213
IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.6004642792452902...
IntegratedHierarchicalHybridRecommender: Fitting complete. Ready to score with beta=0.8911991669986515.
EvaluatorHoldout: Processed 27059 (100.0%) in 12.16 sec. Users per second: 2226
IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.6004642792452902...
IntegratedHierarchicalHybridRecommender: Fitting complete. Ready to score with beta=0.8911991669986515.
EvaluatorHoldout: Processed 27065 (100.0%) in 12.10 sec. Users per second: 2237
IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.6004642792452902...
IntegratedHierarchicalHybridRecommender: Fit

[I 2025-12-24 19:30:26,580] Trial 0 finished with value: 0.2907859912249744 and parameters: {'alpha': 0.6004642792452902, 'beta': 0.8911991669986515}. Best is trial 0 with value: 0.2907859912249744.


[0.29074063255233246, 0.2904299723836878, 0.2917074470613912, 0.2907438726593679, 0.2903080314680926]
IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.6323360661732964...
IntegratedHierarchicalHybridRecommender: Fitting complete. Ready to score with beta=0.8982107174937978.
EvaluatorHoldout: Processed 27062 (100.0%) in 12.09 sec. Users per second: 2238
IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.6323360661732964...
IntegratedHierarchicalHybridRecommender: Fitting complete. Ready to score with beta=0.8982107174937978.
EvaluatorHoldout: Processed 27059 (100.0%) in 12.09 sec. Users per second: 2239
IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.6323360661732964...
IntegratedHierarchicalHybridRecommender: Fitting complete. Ready to score with beta=0.8982107174937978.
EvaluatorHoldout: Processed 27065 (100.0%) in 12.11 sec. Users per second: 2235
IntegratedHierarchicalHybridRecommender: Cr

[I 2025-12-24 19:31:29,630] Trial 1 finished with value: 0.2909282606883521 and parameters: {'alpha': 0.6323360661732964, 'beta': 0.8982107174937978}. Best is trial 1 with value: 0.2909282606883521.


...

[0.29130738481757856, 0.2912202125822775, 0.292078828492765, 0.2908192413508096, 0.2906994348397537]
IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.7817051998356976...
IntegratedHierarchicalHybridRecommender: Fitting complete. Ready to score with beta=0.8594040239751498.
EvaluatorHoldout: Processed 27062 (100.0%) in 13.11 sec. Users per second: 2064
IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.7817051998356976...
IntegratedHierarchicalHybridRecommender: Fitting complete. Ready to score with beta=0.8594040239751498.
EvaluatorHoldout: Processed 27059 (100.0%) in 12.32 sec. Users per second: 2196
IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.7817051998356976...
IntegratedHierarchicalHybridRecommender: Fitting complete. Ready to score with beta=0.8594040239751498.
EvaluatorHoldout: Processed 27065 (100.0%) in 12.39 sec. Users per second: 2184
IntegratedHierarchicalHybridRecommender: Cre

[I 2025-12-24 21:05:18,063] Trial 90 finished with value: 0.29120587539595977 and parameters: {'alpha': 0.7817051998356976, 'beta': 0.8594040239751498}. Best is trial 23 with value: 0.29131409396957575.


[0.290947184953499, 0.29126462740200654, 0.2920712054342725, 0.2909267274248102, 0.2908196317652104]
IntegratedHierarchicalHybridRecommender: Creating hybrid similarity with alpha=0.7909533738281908...
IntegratedHierarchicalHybridRecommender: Fitting complete. Ready to score with beta=0.872018007811915.


[W 2025-12-24 21:05:22,424] Trial 91 failed with parameters: {'alpha': 0.7909533738281908, 'beta': 0.872018007811915} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/wp/jydg89697jzcwnllv2_yz6d40000gn/T/ipykernel_11108/37355673.py", line 46, in objective_function_funksvd
    result, _ = evaluator_test.evaluateRecommender(recommender)
                ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi/Evaluation/Evaluator.py", line 276, in evaluateRecommender
    results_dict = self._run_evaluation_on_selected_users(recommender_object, self.users_to_evaluate)
  File "/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi/Evaluation/Evalua

KeyboardInterrupt: 

# Da qui inizia il training su URM_all

In [ ]:
return

In [ ]:
best_alpha_test = 0.7074505665346519
best_beta_test = 0.9199376036548086
#Trial 8 finished with value: 0.2909141858548001 and parameters: {'alpha': 0.7074505665346519, 'beta': 0.9199376036548086}

In [ ]:
recommender_rp3_f = RP3betaRecommender(URM_all)
recommender_rp3_f.fit(**rp3_params)

In [ ]:
recommender_SLIM_f = SLIMElasticNetRecommender(URM_all)
recommender_SLIM_f.fit(**SLIM_params)

In [ ]:
# Train the final model 
new_similarity = (1 - best_alpha_test) * csr_matrix(recommender_rp3_f.W_sparse) + best_alpha_test * recommender_SLIM_f.W_sparse 

hybridrecommender_object_f = ItemKNNCustomSimilarityRecommender(URM_all)
hybridrecommender_object_f.fit(new_similarity)

In [ ]:
als_recommender_f = FeatureCombinedImplicitALSRecommender(URM_all)
als_recommender_f.fit(**IALS_params)

In [ ]:
recommenders = [hybridrecommender_object_f, als_recommender_f]

linear_comb_rec_f = GeneralizedLinearCoupleHybridRecommender(URM_all, recommenders)
linear_comb_rec_f.fit(best_beta_test)

In [ ]:
import time

start_time = time.time()
results = []
for user_id in df_test_user["user_id"].tolist():
        recommendations = linear_comb_rec_f.recommend(user_id, cutoff=20)       
       
        results.append((user_id, ' '.join(map(str, recommendations))))

df_recommendations = pd.DataFrame(results, columns=["user_id", "item_list"])
print(df_recommendations)
df_recommendations.to_csv("recommendations_ok.csv", index=False)

end_time = time.time()